In [1]:
import deepchem as dc
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import tempfile
import random
import warnings
warnings.filterwarnings("ignore", message="Converting sparse IndexedSlices*", category=UserWarning)

epochs = 300
learning_rate = 1e-3
batch_size = 64
dropout = 0.001

param_grid = {
    'graph_conv_layers': [[64, 64], [128, 64]],
    'dense_layers': [[64], [128]]
}


#param_grid = {             ## for bigger QM8
#    'graph_conv_layers': [[512, 512], [1024, 512]],
#    'dense_layers': [[1024], [2048]]
#}

def df_to_dataset(df_input, tasks):
    featurizer = dc.feat.ConvMolFeaturizer()
    loader = dc.data.CSVLoader(tasks=tasks, smiles_field='SMILES', featurizer=featurizer)
    temp_path = tempfile.mktemp(suffix=".csv")
    df_input.to_csv(temp_path, index=False)
    dataset = loader.create_dataset(temp_path)
    os.remove(temp_path)
    return dataset

tasks = ['Labels']
metric = dc.metrics.Metric(dc.metrics.r2_score, mode='regression')
all_metrics = [
    dc.metrics.Metric(dc.metrics.r2_score, mode='regression'),
    dc.metrics.Metric(dc.metrics.mae_score, mode='regression'),
    dc.metrics.Metric(dc.metrics.rms_score, mode='regression')
]

path = os.path.dirname(os.getcwd())
train_df = pd.read_csv(f'{path}/datasets/example_train.csv')
train_df_filtered = pd.read_csv(f'{path}/datasets/results/clean_3_3.csv').iloc[:,:2]
valid_df = pd.read_csv(f'{path}/datasets/example_valid.csv')
test_df = pd.read_csv(f'{path}/datasets/example_test.csv')

train_dataset = df_to_dataset(train_df, tasks)
train_filtered_dataset = df_to_dataset(train_df_filtered, tasks)
valid_dataset = df_to_dataset(valid_df, tasks)
test_dataset = df_to_dataset(test_df, tasks)

train_dataset_dict = {'original': train_dataset, 'filtered': train_filtered_dataset}
all_results_sole = []

for dataset_type, current_train_dataset in train_dataset_dict.items():
    print(f"\n===== Training with {dataset_type} dataset =====")

    np.random.seed(42)
    tf.random.set_seed(42)
    random.seed(42)

    best_val_r2 = -np.inf
    best_model = None
    best_params = None

    for conv_layers in param_grid['graph_conv_layers']:
        for dense_layers in param_grid['dense_layers']:
            model_dir = f"{path}/datasets/results/graphconv_models/{dataset_type}_conv{'-'.join(map(str, conv_layers))}_dense{'-'.join(map(str, dense_layers))}"

            model = dc.models.GraphConvModel(
                n_tasks=len(tasks),
                graph_conv_layers=conv_layers,
                dense_layers=dense_layers,
                learning_rate=learning_rate,
                batch_size=batch_size,
                dropout=dropout,
                mode='regression',
                model_dir=model_dir,
                random_seed=42,
                verbose=False
            )

            model.fit(current_train_dataset, nb_epoch=epochs)
            val_pred = model.predict(valid_dataset)

            val_r2 = model.evaluate(valid_dataset, [metric], transformers=[])['r2_score']
            print(f"→ GraphConv(conv={conv_layers}, dense={dense_layers}) → Val R² = {val_r2:.4f}")

            if val_r2 > best_val_r2:
                best_val_r2 = val_r2
                best_model = model
                best_params = {
                    'graph_conv_layers': conv_layers,
                    'dense_layers': dense_layers,
                    'learning_rate': learning_rate,
                    'batch_size': batch_size,
                    'dropout': dropout
                }

    train_scores = best_model.evaluate(current_train_dataset, all_metrics, transformers=[])
    valid_scores = best_model.evaluate(valid_dataset, all_metrics, transformers=[])
    test_scores = best_model.evaluate(test_dataset, all_metrics, transformers=[])

    print(f"Dataset: {dataset_type}")
    print(f"Train: {train_scores}")
    print(f"Valid: {valid_scores}")
    print(f"Test : {test_scores}")

    all_results_sole.append({
        'dataset_type': dataset_type,
        'used_params': best_params,
        'train_scores': train_scores,
        'valid_scores': valid_scores,
        'test_scores': test_scores
    })

results_df = pd.DataFrame([{
    'dataset_type': res['dataset_type'],
    **res['used_params'],
    'train_r2': res['train_scores']['r2_score'],
    'train_mae': res['train_scores']['mae_score'],
    'train_rmse': res['train_scores']['rms_score'],
    'valid_r2': res['valid_scores']['r2_score'],
    'valid_mae': res['valid_scores']['mae_score'],
    'valid_rmse': res['valid_scores']['rms_score'],
    'test_r2': res['test_scores']['r2_score'],
    'test_mae': res['test_scores']['mae_score'],
    'test_rmse': res['test_scores']['rms_score']
} for res in all_results_sole])

results_df.to_csv(f"{path}/datasets/results/GC_results.csv", index=False)

smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.
smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.
smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.
smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.
2026-07-18 18:13:58.565203: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.



===== Training with original dataset =====


2026-07-18 18:13:59.157064: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1510] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21977 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090 Ti, pci bus id: 0000:41:00.0, compute capability: 8.6
2026-07-18 18:14:02.296510: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)
2026-07-18 18:14:03.497693: I tensorflow/stream_executor/cuda/cuda_blas.cc:1760] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


→ GraphConv(conv=[64, 64], dense=[64]) → Val R² = 0.5255
→ GraphConv(conv=[64, 64], dense=[128]) → Val R² = 0.5608
→ GraphConv(conv=[128, 64], dense=[64]) → Val R² = 0.5518
→ GraphConv(conv=[128, 64], dense=[128]) → Val R² = 0.5230
Dataset: original
Train: {'r2_score': 0.975045132617732, 'mae_score': 0.3722605599296296, 'rms_score': 0.7377566025820457}
Valid: {'r2_score': 0.5607871835232147, 'mae_score': 1.7178049258244914, 'rms_score': 2.922500507442148}
Test : {'r2_score': 0.5988858508928363, 'mae_score': 1.6512828221321105, 'rms_score': 2.5917113624625374}

===== Training with filtered dataset =====
→ GraphConv(conv=[64, 64], dense=[64]) → Val R² = 0.5510
→ GraphConv(conv=[64, 64], dense=[128]) → Val R² = 0.5707
→ GraphConv(conv=[128, 64], dense=[64]) → Val R² = 0.5648
→ GraphConv(conv=[128, 64], dense=[128]) → Val R² = 0.6175
Dataset: filtered
Train: {'r2_score': 0.9747449456053128, 'mae_score': 0.34082956436719053, 'rms_score': 0.7157730189407467}
Valid: {'r2_score': 0.61750912186